# The Carbon Credit Illusion
## An Energy Balance Framework for Forest-Based Climate Interventions

Carbon credit markets promise a simple deal: you emit 1 tCO₂, you pay someone to plant trees, the trees absorb 1 tCO₂, the climate is unaffected. But this rests on assumptions that physics doesn't support.

This notebook builds up the physical argument step by step, starting from the naive credit promise and progressively adding terms to the planetary energy balance. Each step reveals another way the naive promise falls short — and by how much.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from nbs.legacy.parameters import T_START, T_END, COMPLEXITY_LEVELS
from nbs.legacy.carbon_module import compute_nee_at_level, compute_biomass_growth
from nbs.legacy.albedo_module import compute_albedo_forcing_timeseries
from nbs.legacy.et_module import compute_et_forcing_timeseries
from nbs.fair_module import run_fair_baseline, compute_pulse_warming
from nbs.legacy.credit_analysis import run_at_level, run_progressive_analysis, compute_equivalence_ratio
from nbs.legacy.credit_figures import LEVEL_COLORS, LEVEL_LABELS

YEARS = np.arange(T_START, T_END)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

---
## Part 1: The Naive Promise — How Many Trees to Save the Planet?

Let's start where carbon markets do. The question is simple: how much CO₂ do we need to remove, and how many trees does that take?

### The Carbon Budget

To limit global warming to 1.5°C, the IPCC estimates a remaining carbon budget of roughly **300 GtCO₂** from 2025 onwards (IPCC AR6 WG1, Table SPM.2). Every tCO₂ we emit must either be reduced or compensated.

### A Tree's Carbon Capacity

A mature tropical tree holds approximately 22 kg CO₂ in its biomass. Scaling up:
- 1 hectare of tropical forest stores ~730 tCO₂ (above and below ground)
- At a typical planting density of 1,000 trees/ha: ~0.73 tCO₂/tree

### The Static Calculation

In [2]:
# IPCC AR6 remaining 1.5°C carbon budget (GtCO2, from 2025)
budget_gtco2 = 300.0

# Tropical forest carbon storage parameters (from our model)
B_max_tC_per_ha = 200.0       # max biomass [tC/ha], Chapman-Richards
B_max_tCO2_per_ha = B_max_tC_per_ha * (44/12)  # [tCO2/ha]
planting_density = 1000       # trees/ha
co2_per_tree = B_max_tCO2_per_ha / planting_density  # tCO2/tree at maturity

# Static calculation
trees_needed = (budget_gtco2 * 1e9) / co2_per_tree
area_needed_ha = trees_needed / planting_density
area_needed_mkm2 = area_needed_ha / 1e10  # million km²

print("=== The Naive Static Calculation ===")
print(f"Remaining budget:      {budget_gtco2:.0f} GtCO₂")
print(f"CO₂ per mature tree:   {co2_per_tree:.2f} tCO₂/tree")
print(f"Trees needed:          {trees_needed/1e12:.1f} trillion trees")
print(f"Land area needed:      {area_needed_ha/1e9:.1f} billion ha = {area_needed_mkm2:.0f} million km²")
print(f"  (for reference, total global land area: ~150 million km²)")
print()
print("The naive answer: plant ~{:.0f} trillion trees, done.".format(trees_needed/1e12))
print("But this assumes trees work instantly and identically everywhere.")
print("Let's examine what physics actually says.")

=== The Naive Static Calculation ===
Remaining budget:      300 GtCO₂
CO₂ per mature tree:   0.73 tCO₂/tree
Trees needed:          0.4 trillion trees
Land area needed:      0.4 billion ha = 0 million km²
  (for reference, total global land area: ~150 million km²)

The naive answer: plant ~0 trillion trees, done.
But this assumes trees work instantly and identically everywhere.
Let's examine what physics actually says.


---
## Part 2: The Physics — Energy Balance

### Warming = Accumulated Energy Imbalance

Climate warming isn't about CO₂ concentration alone. It's about **energy**. The planet warms when more energy arrives from the sun than leaves as infrared radiation — an energy imbalance.

The global energy balance at the top of the atmosphere:

$$\frac{dE}{dt} = F_{\text{in}} - F_{\text{out}}$$

where:
$$F_{\text{in}} = \frac{(1 - \alpha) S_0}{4}$$  
$$F_{\text{out}} = \text{OLR} - \Delta F_{\text{CO}_2} - \Delta F_{\text{other}}$$

Cumulative warming is the **integral** of this imbalance over time. This is crucial: a brief spike of forcing causes less warming than a sustained one.

### What Forests Change

Forests don't just store carbon. They modify **multiple terms** of the energy balance:

| Term | Process | Sign | Magnitude |
|------|---------|------|----------|
| $\Delta F_{\text{CO}_2}$ | Carbon uptake reduces atmospheric CO₂ → less GHG forcing | Cooling | Major |
| $\Delta F_\alpha$ | Dark forest canopy absorbs more sunlight than bare land | Warming | Large in boreal |
| $\Delta F_{\text{ET}}$ | Evapotranspiration cools surface via latent heat flux | Cooling | Large in tropics |
| $\Delta F_{\text{soil}}$ | Soil respiration releases CO₂ (Q10: accelerates with warming) | Warming | Gradual |
| $\Delta F_{\text{SOC}}$ | Land conversion can release or gain soil organic carbon | Either | Initial spike |

### A Note on Evapotranspiration

Evapotranspiration (ET) is often overlooked. When forests transpire water:  
1. **Evaporation** at the surface **absorbs** latent heat → surface cools  
2. Water vapour rises and **condenses** in the troposphere, **releasing** that latent heat → vertical redistribution  
3. Net global effect: primarily works through water vapour feedback (H₂O is itself a GHG), cloud formation, and reduced outgoing longwave from cooler surface  

> **Note:** Condensation *releases* energy (exothermic) — it does not absorb it. The surface *cooling* comes from the evaporation step. The atmosphere is then warmed at the condensation level. Globally, the strongest effects are: (a) surface cooling in tropics, and (b) increased water vapour as a greenhouse gas.

Parameterised from observations (Bonan 2008; Duveiller et al. 2018):  
- Tropical forests: −1.5 W/m² (strong cooling — ET dominates)  
- Boreal forests: −0.1 W/m² (weak — cold, low evaporation)

### The Key Insight

All these terms interact with each other and with the rest of the Earth system (ocean heat uptake, sea ice, aerosols…). We don't need to model all of that ourselves — the **FaIR v2 climate model** captures the full system response. We'll feed our carbon and forcing terms into FaIR and examine the net temperature impact.

---
## Part 3: Building Up the Energy Balance — Level by Level

We evaluate a thought experiment: **1 GtCO₂ is emitted** (e.g., from fossil fuel combustion). To offset it, **1 Mha of forest** is planted at a given latitude. What happens to global temperature?

At each level, we add one more physical process to the analysis.

### Setup: The Emission Pulse

In [3]:
# Parameters for the analysis
LAT = 45          # latitude [°N] — temperate forest, interesting tradeoffs
SCENARIO = "ssp245"
PULSE_GTCO2 = 1.0
AREA_MHA = 1.0

# Compute the warming from the 1 GtCO2 pulse
pulse_dt = compute_pulse_warming(pulse_gtco2=PULSE_GTCO2, scenario=SCENARIO)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(YEARS, pulse_dt * 1000, color="#d62728", linewidth=2, label="1 GtCO₂ pulse")
ax.axhline(0, color="black", linewidth=0.5, linestyle="--")
ax.fill_between(YEARS, pulse_dt * 1000, 0, where=pulse_dt > 0, alpha=0.15, color="#d62728")
ax.set_xlabel("Year")
ax.set_ylabel("ΔT (mK)")
ax.set_title("Temperature response to 1 GtCO₂ emission pulse (2025)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

warming_debt_pulse = np.sum(np.maximum(pulse_dt, 0)) * 1000
print(f"Peak pulse warming: {pulse_dt.max()*1000:.2f} mK")
print(f"Cumulative warming (K·yr): {np.sum(pulse_dt):.4f}")
print("This warming persists for decades — CO₂ has a long atmospheric lifetime.")

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Peak pulse warming: 0.81 mK
Cumulative warming (K·yr): 0.0957
This warming persists for decades — CO₂ has a long atmospheric lifetime.


/var/folders/l1/k90rhb0j0ns58y35ymznsd700000gn/T/ipykernel_89002/2744018223.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Level 0: The Static Budget (Naive Assumption)

The naive credit promise assumes trees immediately and uniformly absorb CO₂. We model this as a **flat constant NEE** — total carbon capacity spread uniformly over the simulation period. No temporal dynamics, no spatial variation.

This is what a simple carbon accounting model implicitly assumes.

In [4]:
result_L0 = run_at_level(level=0, lat=LAT, scenario=SCENARIO, pulse_gtco2=PULSE_GTCO2, area_mha=AREA_MHA)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# NEE
ax = axes[0]
ax.plot(YEARS, result_L0['nee'], color=LEVEL_COLORS[0], linewidth=2)
ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax.set_xlabel('Year'); ax.set_ylabel('NEE (GtCO₂/yr)')
ax.set_title('L0: Static (flat) carbon uptake')
ax.grid(True, alpha=0.3)

# Temperature impact
ax = axes[1]
ax.plot(YEARS, pulse_dt * 1000, color='#d62728', linewidth=1.5, linestyle='--', label='Pulse (emission)', alpha=0.7)
ax.plot(YEARS, result_L0['offset_dt'] * 1000, color=LEVEL_COLORS[0], linewidth=2, label='L0 offset')
ax.plot(YEARS, result_L0['delta_t'] * 1000, color='black', linewidth=2, label='Net ΔT')
ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
if result_L0['t_star'] != np.inf:
    ax.axvline(result_L0['t_star'], color='green', linewidth=1, linestyle=':', label=f"t* = {result_L0['t_star']:.0f}")
ax.set_xlabel('Year'); ax.set_ylabel('ΔT (mK)')
ax.set_title('L0: Net temperature impact (pulse + flat offset)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"L0 — Equivalence ratio (2100): {result_L0['equivalence_ratio_2100']:.2f}")
print(f"L0 — Warming debt: {result_L0['warming_debt']*1000:.2f} mK·yr")
print(f"L0 — Break-even year: {result_L0['t_star']}")
print()
print("Even with the FLAT (naive) assumption, the emission pulse causes warming")
print("before the forest can absorb it. The equivalence ratio > 1 means you need")
print("more than 1 credit to genuinely neutralise 1 tCO₂.")

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

L0 — Equivalence ratio (2100): 4.70
L0 — Warming debt: 61.61 mK·yr
L0 — Break-even year: 2025.0

Even with the FLAT (naive) assumption, the emission pulse causes warming
before the forest can absorb it. The equivalence ratio > 1 means you need
more than 1 credit to genuinely neutralise 1 tCO₂.


/var/folders/l1/k90rhb0j0ns58y35ymznsd700000gn/T/ipykernel_89002/3096655194.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Level 1: CO₂ Radiative Forcing + Growth Dynamics

**New term: $\Delta F_{\text{CO}_2}(t)$** — but now it evolves over time.

Trees don't grow instantly. The **Chapman-Richards model** describes how biomass accumulates:

$$B(t) = B_{\max} \left(1 - e^{-kt}\right)^p$$

The carbon uptake rate is $\frac{dB}{dt}$ — fast early, slow later as the forest matures. Crucially, the emission pulse causes warming **now**, while the forest absorbs CO₂ **slowly over decades**. This temporal mismatch creates a **warming debt** — excess integrated warming that genuine climate neutrality would require eliminating.

At Level 1 we use tropical growth parameters everywhere (location-independent) to isolate the temporal effect.

In [5]:
# Show the growth curve first
t_years = np.arange(0, 125)
biomass_tropical = compute_biomass_growth(200.0, 0.08, 2.5, t_years)
dnee_dt_tropical = np.diff(biomass_tropical, prepend=0.0) * (44/12)  # dB/dt in tCO2/ha/yr

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.plot(YEARS, biomass_tropical, color='#2ca02c', linewidth=2)
ax.set_xlabel('Year'); ax.set_ylabel('Biomass (tC/ha)')
ax.set_title('Chapman-Richards growth curve (tropical)')
ax.axhline(200.0, color='gray', linestyle='--', alpha=0.5, label='B_max')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(YEARS, dnee_dt_tropical, color='#2ca02c', linewidth=2)
ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax.set_xlabel('Year'); ax.set_ylabel('Carbon uptake rate (tCO₂/ha/yr)')
ax.set_title('Annual CO₂ uptake — fast early, slow later')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

result_L1 = run_at_level(level=1, lat=LAT, scenario=SCENARIO, pulse_gtco2=PULSE_GTCO2, area_mha=AREA_MHA)

fig, ax = plt.subplots(figsize=(10, 5))
ax.fill_between(YEARS, pulse_dt * 1000, 0, where=pulse_dt > 0, alpha=0.15, color='#d62728', label='Warming debt from pulse')
ax.fill_between(YEARS, result_L1['delta_t'] * 1000, 0, where=result_L1['delta_t'] > 0, alpha=0.2, color='orange')
ax.plot(YEARS, pulse_dt * 1000, color='#d62728', linewidth=1.5, linestyle='--', label='Pulse only')
ax.plot(YEARS, result_L1['delta_t'] * 1000, color=LEVEL_COLORS[1], linewidth=2, label='L1: CO₂ forcing + growth curve')
ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
if result_L1['t_star'] != np.inf:
    ax.axvline(result_L1['t_star'], color='green', linewidth=1.5, linestyle=':', label=f"t* = {result_L1['t_star']:.0f}")
ax.set_xlabel('Year'); ax.set_ylabel('ΔT (mK)')
ax.set_title(f'L1: Temporal CO₂ forcing at {LAT}°N — growth curve matters')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"L1 — Equivalence ratio (2100): {result_L1['equivalence_ratio_2100']:.2f}")
print(f"L1 — Warming debt: {result_L1['warming_debt']*1000:.2f} mK·yr")
print(f"L1 — Break-even year: {result_L1['t_star']}")
print(f"\nThe shaded area is 'warming debt': excess warming accumulated before the")
print(f"forest catches up. Even perfect tropical growth doesn't make the credit 1:1.")

/var/folders/l1/k90rhb0j0ns58y35ymznsd700000gn/T/ipykernel_89002/2856131515.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

L1 — Equivalence ratio (2100): 1.95
L1 — Warming debt: 38.06 mK·yr
L1 — Break-even year: 2025.0

The shaded area is 'warming debt': excess warming accumulated before the
forest catches up. Even perfect tropical growth doesn't make the credit 1:1.


/var/folders/l1/k90rhb0j0ns58y35ymznsd700000gn/T/ipykernel_89002/2856131515.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Level 2: Location-Dependent Growth

**New term: latitude modulates $\Delta F_{\text{CO}_2}(t)$**

Growth rates vary dramatically by latitude:

| Latitude | $B_{\max}$ (tC/ha) | $k$ (yr⁻¹) | Description |
|----------|-------------------|-----------|-------------|
| 0–15°N | 200 | 0.08 | Tropical — fast, high capacity |
| 30°N | 120 | 0.05 | Temperate — medium |
| 45–60°N | 100 | 0.03 | Boreal — slow, low capacity |

The same "1 Mha of forest" means very different carbon uptake depending on where you plant it.

In [6]:
growth_params = {
    0:  (200.0, 0.08, 2.5, '#2ca02c', '0°N (Tropical)'),
    30: (120.0, 0.05, 2.0, '#ff7f0e', '30°N (Temperate)'),
    60: (100.0, 0.03, 2.0, '#1f77b4', '60°N (Boreal)'),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for lat_val, (Bmax, k, p, color, label) in growth_params.items():
    biomass = compute_biomass_growth(Bmax, k, p, t_years)
    axes[0].plot(YEARS, biomass, color=color, linewidth=2, label=label)
    dB = np.diff(biomass, prepend=0.0) * (44/12)
    axes[1].plot(YEARS, dB, color=color, linewidth=2, label=label)

for ax, title, ylabel in zip(axes, ['Biomass by latitude', 'Annual CO₂ uptake rate'], ['Biomass (tC/ha)', 'Uptake (tCO₂/ha/yr)']):
    ax.set_xlabel('Year'); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Compare levels 1 vs 2 for our chosen latitude
result_L2 = run_at_level(level=2, lat=LAT, scenario=SCENARIO, pulse_gtco2=PULSE_GTCO2, area_mha=AREA_MHA)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(YEARS, result_L1['delta_t'] * 1000, color=LEVEL_COLORS[1], linewidth=2, linestyle='--', label='L1: Tropical growth (everywhere)', alpha=0.8)
ax.plot(YEARS, result_L2['delta_t'] * 1000, color=LEVEL_COLORS[2], linewidth=2, label=f'L2: Location-dependent ({LAT}°N)')
ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax.set_xlabel('Year'); ax.set_ylabel('ΔT (mK)')
ax.set_title(f'L1 vs L2: Location matters at {LAT}°N')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"L2 ({LAT}°N) — Equivalence ratio (2100): {result_L2['equivalence_ratio_2100']:.2f}")
print(f"L2 ({LAT}°N) — Break-even year: {result_L2['t_star']}")

/var/folders/l1/k90rhb0j0ns58y35ymznsd700000gn/T/ipykernel_89002/2259456168.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

L2 (45°N) — Equivalence ratio (2100): 6.81
L2 (45°N) — Break-even year: 2025.0


/var/folders/l1/k90rhb0j0ns58y35ymznsd700000gn/T/ipykernel_89002/2259456168.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Level 3: + Albedo Forcing ($\Delta F_\alpha$)

**New energy balance term: surface reflectivity**

Forests are darker than open land and snow. The **albedo forcing** measures this extra solar energy absorbed:

$$\Delta F_\alpha = -\frac{S_0}{4} \cdot \Delta\alpha$$

where $\Delta\alpha$ is the change in surface albedo (negative = darker = more energy absorbed = warming).

In tropical regions, forests are already darker than bare soil but the effect is moderate. In **boreal regions**, forests replace highly reflective snow-covered land — this can cause a **positive forcing (warming)** that overwhelms the carbon cooling effect. A boreal carbon credit may actually warm the planet.

Steady-state forcing values (MODIS-derived):
- 0°N tropical: −0.30 W/m² (slight cooling — forest slightly cooler than tropical soil)
- 45°N temperate: +0.80 W/m² (warming — replacing seasonally snowy grassland)
- 60°N boreal: +1.15 W/m² (strong warming — replacing snow-covered tundra)

In [7]:
result_L3 = run_at_level(level=3, lat=LAT, scenario=SCENARIO, pulse_gtco2=PULSE_GTCO2, area_mha=AREA_MHA)

# Show the albedo forcing timeseries
_intervention = 'tropical_reforestation' if LAT <= 15 else 'boreal_afforestation'
albedo_ts = compute_albedo_forcing_timeseries(_intervention, LAT)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(YEARS, albedo_ts, color='#8c564b', linewidth=2)
ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax.fill_between(YEARS, albedo_ts, 0, where=albedo_ts > 0, alpha=0.2, color='red', label='Warming (albedo)')
ax.fill_between(YEARS, albedo_ts, 0, where=albedo_ts < 0, alpha=0.2, color='blue', label='Cooling (albedo)')
ax.set_xlabel('Year'); ax.set_ylabel('ΔF_α (W/m²)')
ax.set_title(f'Albedo forcing at {LAT}°N (3-yr ramp)')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(YEARS, result_L2['delta_t'] * 1000, color=LEVEL_COLORS[2], linewidth=2, linestyle='--', label='L2: CO₂ forcing only', alpha=0.8)
ax.plot(YEARS, result_L3['delta_t'] * 1000, color=LEVEL_COLORS[3], linewidth=2, label='L3: + Albedo')
ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax.set_xlabel('Year'); ax.set_ylabel('ΔT (mK)')
ax.set_title(f'L2 vs L3: Albedo impact at {LAT}°N')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"L3 ({LAT}°N) — Equivalence ratio (2100): {result_L3['equivalence_ratio_2100']:.2f}")
print(f"L3 ({LAT}°N) — Break-even year: {result_L3['t_star']}")
print(f"Albedo forcing at {LAT}°N: {albedo_ts[-1]:.2f} W/m² (positive = warming)")

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

L3 (45°N) — Equivalence ratio (2100): inf
L3 (45°N) — Break-even year: 2025.0
Albedo forcing at 45°N: 0.80 W/m² (positive = warming)


/var/folders/l1/k90rhb0j0ns58y35ymznsd700000gn/T/ipykernel_89002/2993393622.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


#### The Albedo Trap: At High Latitudes, Carbon Credits Can Warm the Planet

In [8]:
fig, ax = plt.subplots(figsize=(12, 5))

for lat_val, color in [(0, '#2ca02c'), (30, '#ff7f0e'), (45, '#1f77b4'), (60, '#9467bd')]:
    res = run_at_level(level=3, lat=lat_val, scenario=SCENARIO, pulse_gtco2=PULSE_GTCO2, area_mha=AREA_MHA)
    t_star_str = f"{res['t_star']:.0f}" if res['t_star'] != np.inf else "never"
    ax.plot(YEARS, res['delta_t'] * 1000, color=color, linewidth=2,
            label=f"{lat_val}°N (t*={t_star_str}, EQ={res['equivalence_ratio_2100']:.1f})")

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Year'); ax.set_ylabel('ΔT (mK)')
ax.set_title('L3: Net temperature impact by latitude (CO₂ + albedo)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("High-latitude forests can cause NET WARMING relative to no intervention!")
print("The albedo warming from replacing snowy land exceeds the CO₂ cooling benefit.")

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

High-latitude forests can cause NET WARMING relative to no intervention!
The albedo warming from replacing snowy land exceeds the CO₂ cooling benefit.


/var/folders/l1/k90rhb0j0ns58y35ymznsd700000gn/T/ipykernel_89002/2159492817.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Level 4: + Evapotranspiration ($\Delta F_{\text{ET}}$)

**New energy balance term: latent heat flux**

Forests pump enormous amounts of water through transpiration. This latent heat flux is always **negative** (cooling) — it removes heat from the surface. Unlike albedo, the ET effect is strongest where it's warmest and wettest: tropical forests.

The ET effect partially counteracts the albedo warming at mid-latitudes, and reinforces carbon cooling in the tropics. It follows the growing canopy (10-year ramp to full effect as leaf area develops).

Steady-state values (Bonan 2008; Duveiller et al. 2018):
- 0°N: −1.50 W/m²  
- 30°N: −0.80 W/m²  
- 45°N: −0.40 W/m²  
- 60°N: −0.10 W/m²

In [9]:
result_L4 = run_at_level(level=4, lat=LAT, scenario=SCENARIO, pulse_gtco2=PULSE_GTCO2, area_mha=AREA_MHA)

et_ts = compute_et_forcing_timeseries(_intervention, LAT)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(YEARS, et_ts, color='#17becf', linewidth=2, label='ET forcing')
ax.plot(YEARS, albedo_ts, color='#8c564b', linewidth=2, linestyle='--', label='Albedo forcing', alpha=0.8)
ax.plot(YEARS, albedo_ts + et_ts, color='black', linewidth=2, label='Combined non-CO₂')
ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax.set_xlabel('Year'); ax.set_ylabel('Forcing (W/m²)')
ax.set_title(f'Non-CO₂ forcings at {LAT}°N')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(YEARS, result_L3['delta_t'] * 1000, color=LEVEL_COLORS[3], linewidth=2, linestyle='--', label='L3: + Albedo', alpha=0.8)
ax.plot(YEARS, result_L4['delta_t'] * 1000, color=LEVEL_COLORS[4], linewidth=2, label='L4: + ET')
ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax.set_xlabel('Year'); ax.set_ylabel('ΔT (mK)')
ax.set_title(f'L3 vs L4: ET effect at {LAT}°N')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"L4 ({LAT}°N) — Equivalence ratio (2100): {result_L4['equivalence_ratio_2100']:.2f}")
print(f"ET forcing at {LAT}°N: {et_ts[-1]:.2f} W/m² (cooling)")
print(f"Albedo forcing at {LAT}°N: {albedo_ts[-1]:.2f} W/m²")
print(f"Net non-CO₂ forcing: {(et_ts + albedo_ts)[-1]:.2f} W/m²")

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

L4 (45°N) — Equivalence ratio (2100): inf
ET forcing at 45°N: -0.30 W/m² (cooling)
Albedo forcing at 45°N: 0.80 W/m²
Net non-CO₂ forcing: 0.50 W/m²


/var/folders/l1/k90rhb0j0ns58y35ymznsd700000gn/T/ipykernel_89002/3872463468.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Level 5: + Soil Dynamics (SOC + Q10 Respiration)

**New carbon cycle terms: soil organic carbon and temperature feedback**

Two hidden carbon flows complicate the picture:

**1. Soil Organic Carbon (SOC) dynamics**  
Converting land to forest changes the equilibrium SOC. If forest SOC equilibrium > initial SOC (e.g., converting degraded land), the soil gradually *gains* carbon — an additional cooling benefit. If forest SOC < initial SOC (e.g., converting carbon-rich grassland), the soil *releases* carbon — an initial warming spike.

$$\text{SOC}(t) = \text{SOC}_{\text{forest}} + (\text{SOC}_{\text{initial}} - \text{SOC}_{\text{forest}}) \cdot e^{-\mu t}$$

**2. Q10 Soil Respiration Warming Feedback**  
Soil microbes respire faster at higher temperatures. As the world warms, soil respiration increases:

$$R_{\text{soil}}(T) = R_{\text{base}} \cdot Q_{10}^{(T - T_0)/10}$$

We count only the *warming-induced excess* (above baseline respiration), which represents a positive CO₂ feedback: more warming → more soil CO₂ → more warming.

In [10]:
result_L5 = run_at_level(level=5, lat=LAT, scenario=SCENARIO, pulse_gtco2=PULSE_GTCO2, area_mha=AREA_MHA)

nee_L2 = run_at_level(level=2, lat=LAT, scenario=SCENARIO, pulse_gtco2=PULSE_GTCO2, area_mha=AREA_MHA)['nee']
nee_L5 = result_L5['nee']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(YEARS, nee_L2, color=LEVEL_COLORS[2], linewidth=2, linestyle='--', label='L2: biomass only', alpha=0.8)
ax.plot(YEARS, nee_L5, color=LEVEL_COLORS[5], linewidth=2, label='L5: + SOC + Q10')
ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax.set_xlabel('Year'); ax.set_ylabel('NEE (GtCO₂/yr)')
ax.set_title(f'NEE with soil dynamics at {LAT}°N')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(YEARS, result_L4['delta_t'] * 1000, color=LEVEL_COLORS[4], linewidth=2, linestyle='--', label='L4: + ET', alpha=0.8)
ax.plot(YEARS, result_L5['delta_t'] * 1000, color=LEVEL_COLORS[5], linewidth=2, label='L5: + Soil dynamics')
ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax.set_xlabel('Year'); ax.set_ylabel('ΔT (mK)')
ax.set_title(f'L4 vs L5: Soil effects at {LAT}°N')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"L5 ({LAT}°N) — Equivalence ratio (2100): {result_L5['equivalence_ratio_2100']:.2f}")
print(f"L5 ({LAT}°N) — Break-even year: {result_L5['t_star']}")

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

L5 (45°N) — Equivalence ratio (2100): inf
L5 (45°N) — Break-even year: 2025.0


/var/folders/l1/k90rhb0j0ns58y35ymznsd700000gn/T/ipykernel_89002/3027267514.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Level 6: Full Earth System — FaIR Climate Model

**New: disturbance risk + full Earth system interactions**

All the terms we've added — CO₂ forcing, albedo, ET, soil respiration — don't act in isolation. They interact with the full Earth system: **ocean heat uptake, sea ice albedo feedback, aerosol interactions, methane chemistry, stratospheric water vapour…**

We don't model these ourselves. Instead, we feed all our forcing terms into **FaIR v2** (Leach et al. 2021) — a simple climate model calibrated to the CMIP6 ensemble and used in IPCC AR6.

At Level 6, we also add **disturbance risk** — the probability that fire, pest, or drought kills the forest before maturity. This probability rises linearly with warming:

$$p_{\text{disturb}}(t) = p_0 \cdot \left(1 + 0.5 \cdot \frac{t - 2025}{125}\right)$$

FaIR captures how our perturbations propagate through ocean heat uptake, feedback loops, and the carbon cycle.

In [11]:
result_L6 = run_at_level(level=6, lat=LAT, scenario=SCENARIO, pulse_gtco2=PULSE_GTCO2, area_mha=AREA_MHA)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(YEARS, result_L5['nee'], color=LEVEL_COLORS[5], linewidth=2, linestyle='--', label='L5: no disturbance', alpha=0.8)
ax.plot(YEARS, result_L6['nee'], color=LEVEL_COLORS[6], linewidth=2, label='L6: + disturbance')
ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax.set_xlabel('Year'); ax.set_ylabel('NEE (GtCO₂/yr)')
ax.set_title(f'NEE with disturbance at {LAT}°N')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(YEARS, result_L5['delta_t'] * 1000, color=LEVEL_COLORS[5], linewidth=2, linestyle='--', label='L5: soil dynamics', alpha=0.8)
ax.plot(YEARS, result_L6['delta_t'] * 1000, color=LEVEL_COLORS[6], linewidth=2, label='L6: Full model')
ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax.set_xlabel('Year'); ax.set_ylabel('ΔT (mK)')
ax.set_title(f'L5 vs L6: Full Earth system at {LAT}°N')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"L6 ({LAT}°N) — Equivalence ratio (2100): {result_L6['equivalence_ratio_2100']:.2f}")
print(f"L6 ({LAT}°N) — Break-even year: {result_L6['t_star']}")

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

L6 (45°N) — Equivalence ratio (2100): inf
L6 (45°N) — Break-even year: 2025.0


/var/folders/l1/k90rhb0j0ns58y35ymznsd700000gn/T/ipykernel_89002/4246905985.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Part 4: The Full Picture — All Levels Together

In [12]:
print(f"Running all levels for {LAT}°N...")
progressive = run_progressive_analysis(lat=LAT, scenario=SCENARIO,
                                        pulse_gtco2=PULSE_GTCO2, area_mha=AREA_MHA)

fig, ax = plt.subplots(figsize=(12, 6))

for level in COMPLEXITY_LEVELS:
    data = progressive[level]
    lw = 2.5 if level in (0, 6) else 1.5
    alpha = 1.0 if level in (0, 6) else 0.75
    t_star_str = f"{data['t_star']:.0f}" if data['t_star'] != np.inf else "∞"
    eq_str = f"{data['equivalence_ratio_2100']:.1f}" if not np.isinf(data['equivalence_ratio_2100']) else "∞"
    label = f"{LEVEL_LABELS[level]} | t*={t_star_str} EQ={eq_str}"
    ax.plot(YEARS, data['delta_t'] * 1000, color=LEVEL_COLORS[level],
            linewidth=lw, alpha=alpha, label=label)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Year'); ax.set_ylabel('ΔT (mK)')
ax.set_title(f'Progressive energy balance: {LAT}°N, {AREA_MHA} Mha vs {PULSE_GTCO2} GtCO₂ pulse')
ax.legend(fontsize=8, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n{'Level':<6} {'Name':<35} {'Equiv. Ratio (2100)':<22} {'Warming Debt (mK·yr)':<22} {'Break-even'}")
print("-" * 95)
for level in COMPLEXITY_LEVELS:
    d = progressive[level]
    eq = f"{d['equivalence_ratio_2100']:.2f}" if not np.isinf(d['equivalence_ratio_2100']) else "∞"
    ts = f"{d['t_star']:.0f}" if d['t_star'] != np.inf else "never"
    print(f"L{level:<5} {LEVEL_LABELS[level]:<35} {eq:<22} {d['warming_debt']*1000:<22.2f} {ts}")

Running all levels for 45°N...


Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

... (7 outputs left out)

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]


{'Level':<6} {'Name':<35} {'Equiv. Ratio (2100)':<22} {'Warming Debt (mK·yr)':<22} {'Break-even'}
-----------------------------------------------------------------------------------------------
L0     L0: Static budget (naive)           4.70                   61.61                  2025
L1     L1: CO₂ forcing + growth curve      1.95                   38.06                  2025
L2     L2: + Location-dependent growth     6.81                   74.82                  2025
L3     L3: + Albedo (ΔF_α)                 ∞                      87146.63               2025
L4     L4: + Evapotranspiration (ΔF_ET)    ∞                      55655.41               2025
L5     L5: + Soil dynamics (SOC + Q10)     ∞                      55661.90               2025
L6     L6: Full Earth system (FaIR)        ∞                      55673.22               2025


/var/folders/l1/k90rhb0j0ns58y35ymznsd700000gn/T/ipykernel_89002/1456625686.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Part 5: Equivalence Ratio by Latitude

How many credits does it really take to offset 1 tCO₂, as a function of where the forest is planted?

In [13]:
latitudes = [0, 15, 30, 45, 60]
levels_to_show = [1, 3, 4, 6]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for level in levels_to_show:
    eq_ratios = []
    valid_lats = []
    for lat_val in latitudes:
        r = run_at_level(level, lat_val, scenario=SCENARIO, pulse_gtco2=PULSE_GTCO2, area_mha=AREA_MHA)
        if not np.isinf(r['equivalence_ratio_2100']):
            valid_lats.append(lat_val)
            eq_ratios.append(r['equivalence_ratio_2100'])
    if valid_lats:
        ax.plot(valid_lats, eq_ratios, marker='o', color=LEVEL_COLORS[level],
                linewidth=2, label=LEVEL_LABELS[level])

ax.axhline(1, color='black', linewidth=1, linestyle='--', label='Naive 1:1 assumption')
ax.set_xlabel('Latitude (°N)'); ax.set_ylabel('Equivalence ratio (2100)')
ax.set_title('Credits needed per tCO₂ by latitude and complexity')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[1]
for level in levels_to_show:
    t_stars = []
    valid_lats = []
    for lat_val in latitudes:
        r = run_at_level(level, lat_val, scenario=SCENARIO, pulse_gtco2=PULSE_GTCO2, area_mha=AREA_MHA)
        if r['t_star'] != np.inf:
            valid_lats.append(lat_val)
            t_stars.append(r['t_star'])
    if valid_lats:
        ax.plot(valid_lats, t_stars, marker='o', color=LEVEL_COLORS[level],
                linewidth=2, label=LEVEL_LABELS[level])

ax.set_xlabel('Latitude (°N)'); ax.set_ylabel('Break-even year (t*)')
ax.set_title('When does the intervention become net cooling?')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey finding: the naive 1:1 credit assumption (dashed line) is NEVER correct.")
print("Physical reality requires anywhere from 2× to ∞ credits for genuine climate neutrality,")
print("depending on latitude and which physical processes you include.")

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

... (72 outputs left out)

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]


Key finding: the naive 1:1 credit assumption (dashed line) is NEVER correct.
Physical reality requires anywhere from 2× to ∞ credits for genuine climate neutrality,
depending on latitude and which physical processes you include.


/var/folders/l1/k90rhb0j0ns58y35ymznsd700000gn/T/ipykernel_89002/1922751483.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Part 6: The Role of the Background Climate Scenario

Everything we've done so far assumed SSP2-4.5 (intermediate emissions). Does the background scenario matter?

In [14]:
scenarios = {'ssp126': '#4daf4a', 'ssp245': '#377eb8', 'ssp370': '#e41a1c'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_idx, level in enumerate([3, 6]):
    ax = axes[ax_idx]
    for scenario, color in scenarios.items():
        eq_ratios = []
        valid_lats = []
        for lat_val in latitudes:
            r = run_at_level(level, lat_val, scenario=scenario, pulse_gtco2=PULSE_GTCO2, area_mha=AREA_MHA)
            if not np.isinf(r['equivalence_ratio_2100']):
                valid_lats.append(lat_val)
                eq_ratios.append(r['equivalence_ratio_2100'])
        if valid_lats:
            ax.plot(valid_lats, eq_ratios, marker='o', color=color,
                    linewidth=2, label=scenario.upper())
    ax.axhline(1, color='black', linewidth=1, linestyle='--', label='1:1 naive')
    ax.set_xlabel('Latitude (°N)'); ax.set_ylabel('Equivalence ratio (2100)')
    ax.set_title(f'{LEVEL_LABELS[level]}')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.suptitle('Equivalence ratio under SSP scenarios', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

... (61 outputs left out)

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

Running 1 projections in parallel:   0%|          | 0/400 [00:00<?, ?timesteps/s]

/var/folders/l1/k90rhb0j0ns58y35ymznsd700000gn/T/ipykernel_89002/2741551836.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
plt.savefig('equivalence_ratio_by_latitude_and_scenario.png', dpi=300, bbox_inches='tight')

---
## Summary: What Physics Says About Carbon Credits

Starting from the naive static carbon budget and building up the energy balance term by term:

| Physical process | Direction | Magnitude | Latitude dependence |
|---|---|---|---|
| CO₂ uptake (growth curve) | Cooling | Large | Weak |
| Temporal delay | Warming debt | Moderate | Moderate |
| Location-dependent growth | Slows offset | Large | Strong |
| Albedo ($\Delta F_\alpha$) | Warming (boreal) / slight cooling (tropical) | Very large in boreal | Dominant at high lat |
| Evapotranspiration ($\Delta F_{\text{ET}}$) | Cooling | Moderate | Strongest tropical |
| Soil respiration (Q10) | Warming (feedback) | Small–moderate | Moderate |
| SOC dynamics | Either | Small | Moderate |
| Disturbance risk | Warming | Small, growing | Moderate |

### The Bottom Line

1. **The 1:1 credit ratio is never physically justified.** Even with the most optimistic assumptions, temporal delay alone requires >1 credit per tCO₂.

2. **Location matters enormously.** Tropical forests are the most effective by far. Boreal afforestation can warm the planet faster than it cools it.

3. **ET adds a significant cooling benefit** that partially compensates albedo warming, especially in tropical regions.

4. **Soil processes add another layer of complexity** — and mostly make the picture worse.

5. **The full picture requires a climate model.** Individual forcing terms interact nonlinearly through ocean heat uptake, water vapour feedbacks, and carbon cycle dynamics. FaIR captures these interactions.

**The naive promise of carbon credits — plant trees, neutralise emissions — requires multiple physical corrections before it approaches reality.**